In [14]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim


import torch

import torch.nn.functional as F

from scipy.stats import spearmanr

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
import nltk

from nltk.corpus import wordnet as wn

import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

from wordfreq import top_n_list



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12708.89it/s]


In [3]:
#filtro per ottenere solo parole alfabetiche
vocab = [
    w for w in top_n_list("en", 120000)
    if w.isalpha()
    if len(w)  >= 3
]

In [4]:
vocab_100k = vocab[:100000]

In [2]:
#per aprire vocab 100k
with open("vocab_100k.txt", "r", encoding="utf-8") as f:
    vocab_100k = [line.strip() for line in f]

FileNotFoundError: [Errno 2] No such file or directory: 'vocab_100k.txt'

In [5]:
#calcolo gli embedding una sola volta grazie a vocab 100kk
embeddings_matrix = model.encode(
    vocab_100k,
    convert_to_tensor=True,
    batch_size=128,
    show_progress_bar=True
)

print("Embedding creati:", len(embeddings_matrix))

Batches:   2%|▏         | 15/782 [00:02<02:32,  5.02it/s]


KeyboardInterrupt: 

In [17]:
# salvo La matrice per non ricalcolare gli embedding ogni volta
torch.save(
    {
        "vocab": vocab_100k,
        "embeddings": embeddings_matrix.cpu()
    },
    "sbert_vocab_100k.pt"
)

print("Salvataggio completato!")

Salvataggio completato!


In [6]:
#codice da eseguire per avere la matrice di embedding
data = torch.load("sbert_vocab_100k.pt")

vocab_100k = data["vocab"]
embeddings_matrix = data["embeddings"]

In [7]:
#associo parola a embedding perchè prima ho la matrice
embeddings = {
    word: embedding
    for word, embedding in zip(vocab_100k, embeddings_matrix)
}

print("Embedding creati:", len(embeddings))

Embedding creati: 100000


In [8]:
#carico i dataset
Analogie_df = pd.read_csv("Analogie.csv")

In [9]:
#anche per 50k e 30k
vocab_30k = vocab_100k[:30000]
vocab_50k = vocab_100k[:50000]

embeddings_30k = embeddings_matrix[:30000]
embeddings_50k = embeddings_matrix[:50000]

print(len(embeddings_30k))
print(len(embeddings_50k))

30000
50000


In [10]:
#anche per loro i dizionari
embeddings_dict_30k = {
    word: embedding
    for word, embedding in zip(vocab_30k, embeddings_30k)
}

embeddings_dict_50k = {
    word: embedding
    for word, embedding in zip(vocab_50k, embeddings_50k)
}

In [11]:
print(Analogie_df.head())

  category        a         b         c           d
0  plurale    river    rivers   science    sciences
1  plurale  science  sciences      song       songs
2  plurale     song     songs    street     streets
3  plurale   street   streets   student    students
4  plurale  student  students  category  categories


In [13]:
a, b, c, d = Analogie_df.iloc[0][["a", "b", "c", "d"]]

vec_a = embeddings[a]
vec_b = embeddings[b]
vec_c = embeddings[c]

result = vec_b - vec_a + vec_c

In [15]:

candidate_vocab = vocab_100k[:30000]

best_word = None
best_score = -1

for word in candidate_vocab:
    # escludiamo le parole già presenti nell'analogia
    if word in [a, b, c]:
        continue

    score = F.cosine_similarity(
        result.unsqueeze(0),
        embeddings[word].unsqueeze(0)
    ).item()

    if score > best_score:
        best_score = score
        best_word = word

print("Analogia:", f"{a} : {b} :: {c} : {d}")
print("Predetto:", best_word)
print("Atteso:", d)
print("Cosine similarity:", best_score)

Analogia: river : rivers :: science : sciences
Predetto: scientific
Atteso: sciences
Cosine similarity: 0.790209174156189


In [18]:
#funzione in generale per testare tutte le analogie
def test_analogy(a, b, c, d, candidate_vocab):
    
    # calcolo del vettore risultante
    result = embeddings[b] - embeddings[a] + embeddings[c]
    
    # cerco la parola più vicina
    best_word = None
    best_score = -1
    
    for word in candidate_vocab:
        
        # escludiamo le parole già presenti nell'analogia
        if word in [a, b, c]:
            continue
        
        score = F.cosine_similarity(
            result.unsqueeze(0),
            embeddings[word].unsqueeze(0)
        ).item()
        
        if score > best_score:
            best_score = score
            best_word = word
    
    # verifico se la risposta è corretta
    correct = (best_word == d)
    
    return best_word, best_score, correct

In [24]:
#risultati per lista 30k
#creo lista di risultati
results_30k = []

for _, row in Analogie_df.iterrows():

    predicted, score, correct = test_analogy(
        row["a"],
        row["b"],
        row["c"],
        row["d"],
        vocab_30k
    )

    results_30k.append({
        "category": row["category"],
        "a": row["a"],
        "b": row["b"],
        "c": row["c"],
        "d": row["d"],
        "predicted": predicted,
        "cosine_similarity": score,
        "correct": correct
    })

df_risultati_30k = pd.DataFrame(results_30k)

In [25]:
print(df_risultati_30k.head())

  category        a         b         c           d   predicted  \
0  plurale    river    rivers   science    sciences  scientific   
1  plurale  science  sciences      song       songs       songs   
2  plurale     song     songs    street     streets     streets   
3  plurale   street   streets   student    students    students   
4  plurale  student  students  category  categories  categories   

   cosine_similarity  correct  
0           0.790209    False  
1           0.653181     True  
2           0.806746     True  
3           0.831288     True  
4           0.851347     True  


In [26]:
#per i 50k
results_50k = []

for _, row in Analogie_df.iterrows():

    predicted, score, correct = test_analogy(
        row["a"],
        row["b"],
        row["c"],
        row["d"],
        vocab_50k
    )

    results_50k.append({
        "category": row["category"],
        "a": row["a"],
        "b": row["b"],
        "c": row["c"],
        "d": row["d"],
        "predicted": predicted,
        "cosine_similarity": score,
        "correct": correct
    })

df_risultati_50k = pd.DataFrame(results_50k)

In [27]:
print(df_risultati_50k.head())

  category        a         b         c           d   predicted  \
0  plurale    river    rivers   science    sciences  scientific   
1  plurale  science  sciences      song       songs       songs   
2  plurale     song     songs    street     streets     streets   
3  plurale   street   streets   student    students    students   
4  plurale  student  students  category  categories  categories   

   cosine_similarity  correct  
0           0.790209    False  
1           0.653181     True  
2           0.806746     True  
3           0.831288     True  
4           0.851347     True  


In [28]:
#per i 100k
results_100k = []

for _, row in Analogie_df.iterrows():

    predicted, score, correct = test_analogy(
        row["a"],
        row["b"],
        row["c"],
        row["d"],
        vocab_100k
    )

    results_100k.append({
        "category": row["category"],
        "a": row["a"],
        "b": row["b"],
        "c": row["c"],
        "d": row["d"],
        "predicted": predicted,
        "cosine_similarity": score,
        "correct": correct
    })

df_risultati_100k = pd.DataFrame(results_100k)

In [29]:
print(df_risultati_100k.head())

  category        a         b         c           d   predicted  \
0  plurale    river    rivers   science    sciences  scientific   
1  plurale  science  sciences      song       songs       songs   
2  plurale     song     songs    street     streets     streets   
3  plurale   street   streets   student    students    students   
4  plurale  student  students  category  categories  categories   

   cosine_similarity  correct  
0           0.790209    False  
1           0.653181     True  
2           0.806746     True  
3           0.831288     True  
4           0.851347     True  


In [31]:
#salvo i dataframe
df_risultati_30k.to_csv("df_risultati_30k.csv", index=False)
df_risultati_50k.to_csv("df_risultati_50k.csv", index=False)
df_risultati_100k.to_csv("df_risultati_100k.csv", index=False)

In [32]:
#codice da eseguire per avere il dataframe
df_risultati_30k = pd.read_csv("df_risultati_30k.csv")
df_risultati_50k = pd.read_csv("df_risultati_50k.csv")
df_risultati_100k = pd.read_csv("df_risultati_100k.csv")

In [36]:
#calcoliamo accuracy per ogni vocab
print(f"30k:  {df_risultati_30k['correct'].value_counts(normalize=True)[True] * 100:.2f}%")
print(f"50k:  {df_risultati_50k['correct'].value_counts(normalize=True)[True] * 100:.2f}%")
print(f"100k: {df_risultati_100k['correct'].value_counts(normalize=True)[True] * 100:.2f}%")

30k:  55.00%
50k:  52.50%
100k: 47.50%


In [ ]:
#tabella accuracy distinta per ogni gruppo di analogia per ogni vocab
tabella_accuracy = pd.DataFrame({
    "30k": df_risultati_30k.groupby("category")["correct"].mean() * 100,
    "50k": df_risultati_50k.groupby("category")["correct"].mean() * 100,
    "100k": df_risultati_100k.groupby("category")["correct"].mean() * 100
})

tabella_accuracy = tabella_accuracy.round(2)

print(tabella_accuracy)

                 30k   50k  100k
category                        
avverbi         30.0  30.0  20.0
capitale_paese  60.0  60.0  60.0
genere          50.0  40.0  30.0
plurale         80.0  80.0  80.0
